In [1]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine , text


In [22]:
# Read CSV
df = pd.read_csv("pd.read_csv("data/enrolments.csv"))

# Datatype conversion 

# we have mixed date format in the course date field - convert course date to date field

df["course_date"] = pd.to_datetime(df["course_date"], errors = "coerce", format ="mixed")
df["course_date"] = df["course_date"].dt.date
# df.info()

In [23]:
# if the python verion doesn't support format = "mixed"
# from dateutil.parser import parse

# def parse_date(x):
#     try:
#         return parse(str(x), dayfirst=True).date()
#     except:
#         return None

# df["course_date"] = df["course_date"].apply(parse_date)

In [3]:
# To run this code
# Create SQLite database
# Create engine
engine = create_engine("sqlite:///courses.db")

conn = engine.connect()

# conn.execute(text("""drop table if exists courses"""))
conn.execute(text("""drop table if exists enrollments"""))
conn.execute(text("""drop table if exists enrolment"""))
conn.execute(text("""drop table if exists silver_courses"""))
conn.execute(text("""drop table if exists bronze_courses"""))
conn.execute(text("""drop table if exists rejected_enrolments"""))
conn.execute(text("""drop table if exists rejected_courses"""))
conn.execute(text("""drop table if exists enrolments"""))
conn.execute(text("""drop table if exists bronze_enrolments"""))
conn.execute(text("""drop table if exists silver_enrolments"""))

In [4]:
# Create SQLite database
# Create engine
engine = create_engine("sqlite:///courses.db")

conn = engine.connect()

# conn.execute(text("""create table courses (
#     course_id int,
#     name text,
#     description text,
#     prerequisites text )"""
# ));

conn.execute(text("""drop table if exists enrolments"""))
conn.execute(text("""create table enrolments (
    enrollment_id int,
    participant_id string,
    participant_name string,
    course_id int,
    course_date date,
    amount float,
    subsidy float,
    credits_used float
);"""));

#Load data into sqllite
df.to_sql(
    name = "enrolments",
    con = conn,
    if_exists = "replace",
    index = False )
pd.read_sql("""
PRAGMA table_info(courses)
""", engine)

conn.commit()

In [5]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", engine)


,name
0,courses
1,enrolments


In [17]:
pd.read_sql("""
SELECT *
FROM courses
LIMIT 5
""", conn)

,course_id,name,description,prerequisites
0,1,CS100 Introduction to Computer Science,Foundational course covering computational thi...,NaN
1,2,CS101 Data structures,"Study of arrays, linked lists, trees, graphs, ...",NaN
2,3,CS200 Databases,"Covers relational database design, SQL, normal...","[1,2]"
3,4,CS201 Networking,"Explores network protocols, TCP/IP stack, rout...",[3]
4,5,CS300 Systems,"Advanced course on operating systems, concurre...",[4]


In [7]:
#BRONZE LAYER - Ingestion Timestamp ,sourcefile 

conn.execute(text("""
CREATE TABLE bronze_courses AS
SELECT
    *,
    'courses.csv' AS source_file_name,
    CURRENT_TIMESTAMP AS ingestion_timestamp
FROM courses
"""))

conn.execute(text("""
CREATE TABLE bronze_enrolments AS
SELECT
    *,
    'enrolments.csv' AS source_file_name,
    CURRENT_TIMESTAMP AS ingestion_timestamp
FROM enrolments
"""))

conn.commit()
pd.read_sql(
    "select * from bronze_enrolments",engine)

pd.read_sql("""
PRAGMA table_info(bronze_enrolments)
""", engine)

,cid,name,type,notnull,dflt_value,pk
0,0,enrollment_id,INT,0,None,0
1,1,participant_id,TEXT,0,None,0
2,2,participant_name,TEXT,0,None,0
3,3,course_id,INT,0,None,0
4,4,course_date,NUM,0,None,0
5,5,amount,REAL,0,None,0
6,6,subsidy,REAL,0,None,0
7,7,credits_used,REAL,0,None,0
8,8,source_file_name,,0,None,0
9,9,ingestion_timestamp,,0,None,0


In [8]:
conn.execute (text("""
CREATE TABLE silver_courses (
    course_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    description TEXT,
    prerequisites TEXT
)"""))

conn.execute(text("""
CREATE TABLE silver_enrolments (
    enrollment_id INTEGER PRIMARY KEY,
    participant_id TEXT NOT NULL,
    participant_name TEXT NOT NULL,
    course_id INTEGER NOT NULL,
    course_date DATE NOT NULL,
    amount REAL CHECK(amount >= 0),
    subsidy REAL CHECK(subsidy >= 0),
    credits_used REAL CHECK(credits_used >= 0),
    FOREIGN KEY (course_id)
        REFERENCES silver_courses(course_id)
);"""))

conn.execute(text("""
CREATE TABLE rejected_enrolments (
    enrollment_id INTEGER,
    participant_id TEXT,
    participant_name TEXT,
    course_id INTEGER,
    course_date TEXT,
    amount REAL,
    subsidy REAL,
    credits_used REAL,
    rejection_reason TEXT,
    ingestion_timestamp TEXT
);"""))

conn.execute(text("""
CREATE TABLE rejected_courses (
    course_id INTEGER,
    name TEXT,
    description TEXT,
    prerequisites TEXT,
    rejection_reason TEXT,
    ingestion_timestamp TEXT
);"""))

In [9]:
# conn.execute(text("DELETE FROM silver_courses"))
# conn.execute(text("DELETE FROM silver_enrolments"))

conn.execute(text("""
INSERT INTO rejected_courses
SELECT
    course_id,
    name,
    description,
    prerequisites,
    'Invalid prerequisites JSON',
    CURRENT_TIMESTAMP
FROM bronze_courses
WHERE prerequisites IS NOT NULL
  AND json_valid(prerequisites) = 0
"""))

conn.execute(text("""
INSERT INTO silver_courses
SELECT
    course_id,
    TRIM(name),
    TRIM(description),
    prerequisites
FROM bronze_courses
WHERE course_id IS NOT NULL
  AND name IS NOT NULL
  AND (
        prerequisites IS NULL
        OR json_valid(prerequisites) = 1
      )
"""))

In [10]:
conn.execute(text("""
INSERT INTO rejected_enrolments
SELECT
    enrollment_id,
    participant_id,
    participant_name,
    course_id,
    course_date,
    amount,
    subsidy,
    credits_used,
    CASE
        WHEN enrollment_id IS NULL THEN 'Missing enrollment_id'
        WHEN participant_id IS NULL THEN 'Missing participant_id'
        WHEN participant_name IS NULL THEN 'Missing participant_name'
        WHEN course_id IS NULL THEN 'Missing course_id'
        WHEN course_date IS NULL THEN 'Invalid or missing course_date'
        WHEN amount < 0 THEN 'Negative amount'
        WHEN subsidy < 0 THEN 'Negative subsidy'
        WHEN credits_used < 0 THEN 'Negative credits_used'
        
    END,
    CURRENT_TIMESTAMP
FROM bronze_enrolments
WHERE enrollment_id IS NULL
   OR participant_id IS NULL
   OR participant_name IS NULL
   OR course_id IS NULL
   OR course_date IS NULL
   OR amount < 0
   OR subsidy < 0
   OR credits_used < 0
"""))

conn.execute(text("""
INSERT INTO silver_enrolments (
    enrollment_id,
    participant_id,
    participant_name,
    course_id,
    course_date,
    amount,
    subsidy,
    credits_used
)
SELECT DISTINCT
    enrollment_id,
    TRIM(participant_id),
    TRIM(participant_name),
    course_id,
    DATE(course_date),
    CAST(amount AS REAL),
    CAST(subsidy AS REAL),
    CAST(credits_used AS REAL)
FROM bronze_enrolments
WHERE enrollment_id IS NOT NULL
  AND participant_id IS NOT NULL
  AND participant_name IS NOT NULL
  AND course_id IS NOT NULL
  AND course_date IS NOT NULL
  AND amount >= 0
  AND subsidy >= 0
  AND credits_used >= 0
"""))


In [20]:
display(pd.read_sql("select * from silver_courses",conn))
display(pd.read_sql("select * from silver_enrolments",conn))
display(pd.read_sql("select * from rejected_courses",conn))
display(pd.read_sql("select * from rejected_enrolments",conn))

,course_id,name,description,prerequisites
0,1,CS100 Introduction to Computer Science,Foundational course covering computational thi...,NaN
1,2,CS101 Data structures,"Study of arrays, linked lists, trees, graphs, ...",NaN
2,3,CS200 Databases,"Covers relational database design, SQL, normal...","[1,2]"
3,4,CS201 Networking,"Explores network protocols, TCP/IP stack, rout...",[3]
4,5,CS300 Systems,"Advanced course on operating systems, concurre...",[4]


,enrollment_id,participant_id,participant_name,course_id,course_date,amount,subsidy,credits_used
0,1,P001,Alice Tan,1,2024-01-15,500.0,200.0,300.0
1,2,P002,Bob Lee,1,2024-01-15,500.0,250.0,250.0
2,3,P007,Grace Koh,1,2024-01-20,500.0,250.0,250.0
3,4,P009,Irene Chua,1,2024-01-25,500.0,200.0,300.0
4,5,P003,Charlie Ng,2,2024-01-02,600.0,300.0,300.0
5,6,P001,Alice Tan,2,2024-02-01,600.0,200.0,400.0
6,7,P004,Diana Lim,1,2024-02-10,500.0,150.0,350.0
7,8,P007,Grace Koh,2,2024-02-15,600.0,300.0,300.0
8,9,P009,Irene Chua,2,2024-02-20,600.0,250.0,350.0
9,10,P002,Bob Lee,3,2024-03-05,750.0,400.0,350.0


,course_id,name,description,prerequisites,rejection_reason,ingestion_timestamp


,enrollment_id,participant_id,participant_name,course_id,course_date,amount,subsidy,credits_used,rejection_reason,ingestion_timestamp
0,21,P005,Edward Goh,5,None,900.0,500.0,400.0,Invalid or missing course_date,2026-05-31 10:16:07


In [12]:
# try:
#     conn.close()
# except:
#     pass

In [13]:
# Gold Layer
# Prefer star schema to implement this 

# fact table - Enrolments
# dimension table - courses and participant

conn.execute(text("""Drop Table if exists dim_course"""))
conn.execute(text("""Drop Table if exists dim_participant"""))
conn.execute(text("""Drop Table if exists fact_enrolments"""))

conn.execute(text("""CREATE TABLE dim_course AS
SELECT DISTINCT
    course_id,
    name AS course_name,
    description
FROM silver_courses;"""))

conn.execute(text("""CREATE TABLE dim_participant AS
SELECT DISTINCT
    participant_id,
    participant_name
FROM silver_enrolments;"""))

conn.execute(text("""CREATE TABLE fact_enrolments AS
SELECT
    enrollment_id,
    participant_id,
    course_id,
    course_date,
    amount,
    subsidy,
    credits_used
FROM silver_enrolments;"""))


#indexing to improve query performance 

conn.execute(text("""CREATE INDEX idx_fact_course
ON fact_enrolments(course_id);"""))

conn.execute(text("""CREATE INDEX idx_fact_participant
ON fact_enrolments(participant_id);"""))

conn.execute(text("""CREATE INDEX idx_fact_date
ON fact_enrolments(course_date);"""))


#frequently used reports
conn.execute(text("""
DROP TABLE IF EXISTS gold_course_summary
"""))

conn.execute(text("""CREATE TABLE gold_course_summary AS
SELECT
    course_id,
    COUNT(*) AS total_enrolments,
    SUM(amount) AS total_revenue,
    SUM(subsidy) AS total_subsidy,
    SUM(credits_used) AS total_credits_used
FROM silver_enrolments
GROUP BY course_id;"""))

#Monthly trends
conn.execute(text("""
DROP TABLE IF EXISTS gold_monthly_summary
"""))
conn.execute(text("""CREATE TABLE gold_monthly_summary AS
SELECT
    strftime('%Y-%m', course_date) AS month,
    COUNT(*) AS total_enrolments,
    SUM(amount) AS total_revenue
FROM silver_enrolments
GROUP BY month;"""))





In [21]:
display(pd.read_sql("select * from gold_monthly_summary",conn))
display(pd.read_sql("select * from gold_course_summary",conn))

,month,total_enrolments,total_revenue
0,1900-06,1,800.0
1,2024-01,7,4250.0
2,2024-02,4,2300.0
3,2024-03,4,3050.0
4,2024-04,4,2700.0
5,2024-05,4,3300.0
6,2024-06,3,2250.0
7,2024-07,4,3100.0
8,2024-10,2,1250.0


,course_id,total_enrolments,total_revenue,total_subsidy,total_credits_used
0,1,8,4250.0,1800.0,2450.0
1,2,7,4200.0,1850.0,2350.0
2,3,8,6150.0,2750.0,3400.0
3,4,6,4800.0,2200.0,2600.0
4,5,4,3600.0,1700.0,1900.0


In [15]:
#participants that have registered for courses without meeting the prerequisites

pd.read_sql("""WITH prerequisite_courses AS (

    SELECT
        e.enrollment_id,
        e.participant_id,
        e.participant_name,
        e.course_id,
        e.course_date,
        json_each.value AS prerequisite_course_id

    FROM enrolments e

    JOIN courses c
        ON e.course_id = c.course_id

    JOIN json_each(c.prerequisites)

),

missing_prerequisites AS (

    SELECT
        pc.*

    FROM prerequisite_courses pc

    LEFT JOIN enrolments completed

        ON pc.participant_id = completed.participant_id

        AND completed.course_id = pc.prerequisite_course_id

        AND completed.course_date < pc.course_date

    WHERE completed.course_id IS NULL
)

SELECT
    participant_id,
    participant_name,
    course_id,
    prerequisite_course_id AS missing_prerequisite
FROM missing_prerequisites
ORDER BY participant_id, course_id;""",engine)



,participant_id,participant_name,course_id,missing_prerequisite
0,P001,Alice Tan,3,1
1,P001,Alice Tan,3,2
2,P001,Alice Tan,4,3
3,P002,Bob Lee,3,2
4,P005,Edward Goh,4,3
5,P005,Edward Goh,5,4
6,P008,Henry Teo,3,1
7,P008,Henry Teo,3,2
8,P008,Henry Teo,5,4
9,P009,Irene Chua,5,4
